
# OmniLingual 300M Levantine Full — from Whisper split sources

This notebook is configured to **force-rebuild** the full Levantine dataset for `facebook/omniASR-LLM-300M` using the **same raw data paths and split logic used by the Whisper pipeline**, but with the `>=30s` drop rule removed.

It writes:

- `prepare_omnilingual_300m_levantine_full_from_whisper_splits.py`
- `omnilingual_300m_levantine_full_from_whisper_splits.yaml`
- `datasets/omnilingual_300m_levantine_full_from_whisper_splits/{train,validation,test}.jsonl`
- `runs/omnilingual_300m_levantine_full_from_whisper_splits/...`

This is still an Omni export/prepare notebook. Actual OmniLingual fine-tuning remains the external Omni recipe flow unless your generic trainer has native support for this model family.


**Patch note:** this version sets `FORCE_MATERIALIZE = True` and `FORCE_PREPARE = True`, removes old smoke/sample-cap keys from the generated YAML, and validates that the prepared manifests are not accidentally 1-row smoke outputs.


In [1]:
from pathlib import Path
import json
import os
import py_compile
import shutil
import subprocess
import sys

PROJECT_DIR = Path("/home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "omnilingual_300m_levantine_full_from_whisper_splits"
MATERIALIZE_SCRIPT = PROJECT_DIR / "prepare_omnilingual_300m_levantine_full_from_whisper_splits.py"
TRAINER_SCRIPT = PROJECT_DIR / "asr_universal_trainer.py"

BASE_CONFIG_PATH = PROJECT_DIR / "omnilingual_300m_levantine_full.yaml"
CONFIG_PATH = PROJECT_DIR / "omnilingual_300m_levantine_full_from_whisper_splits.yaml"

DATASET_DIR = PROJECT_DIR / "datasets" / DATASET_NAME
WORK_DIR = PROJECT_DIR / "runs" / DATASET_NAME

FORCE_MATERIALIZE = True
FORCE_PREPARE = True

DATASET_REQUIRED_FILES = [
    DATASET_DIR / "dataset_summary.json",
    DATASET_DIR / "train.jsonl",
    DATASET_DIR / "validation.jsonl",
    DATASET_DIR / "test.jsonl",
]
PREPARE_REQUIRED_FILES = [
    WORK_DIR / "prepared" / "train.jsonl",
    WORK_DIR / "prepared" / "validation.jsonl",
    WORK_DIR / "prepared" / "test.jsonl",
    WORK_DIR / "prepared" / "stats.json",
    WORK_DIR / "prepared" / "dataset_resolved.json",
    WORK_DIR / "prepared" / "model_spec.json",
]


def outputs_exist(paths):
    return all(path.exists() for path in paths)



def missing_outputs(paths):
    return [path for path in paths if not path.exists()]



def write_text_if_changed(path: Path, text: str, *, encoding: str = "utf-8") -> bool:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_text(encoding=encoding) == text:
        print("Unchanged:", path)
        return False
    path.write_text(text, encoding=encoding)
    print("Wrote:", path)
    return True



def describe_outputs(label: str, paths):
    missing = missing_outputs(paths)
    if not missing:
        print(f"{label}: ready")
        return
    print(f"{label}: missing {len(missing)} file(s)")
    for item in missing:
        print(" -", item)


print("Project dir:", PROJECT_DIR)
print("Dataset name:", DATASET_NAME)
print("Materialize script:", MATERIALIZE_SCRIPT)
print("Trainer script:", TRAINER_SCRIPT)
print("Base config:", BASE_CONFIG_PATH)
print("New config:", CONFIG_PATH)
print("Dataset dir:", DATASET_DIR)
print("Work dir:", WORK_DIR)
print("FORCE_MATERIALIZE:", FORCE_MATERIALIZE)
print("FORCE_PREPARE:", FORCE_PREPARE)
describe_outputs("Dataset artifacts", DATASET_REQUIRED_FILES)
describe_outputs("Prepare artifacts", PREPARE_REQUIRED_FILES)


Project dir: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer
Dataset name: omnilingual_300m_levantine_full_from_whisper_splits
Materialize script: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/prepare_omnilingual_300m_levantine_full_from_whisper_splits.py
Trainer script: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/asr_universal_trainer.py
Base config: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/omnilingual_300m_levantine_full.yaml
New config: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/omnilingual_300m_levantine_full_from_whisper_splits.yaml
Dataset dir: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/datasets/omnilingual_300m_levantine_full_from_whisper_splits
Work dir: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/runs/omnilingual_300m_levantine_full_from_whisper_splits
FORCE_MATERIALIZE: True
FORCE_PREPARE: True
Dataset artifacts: ready
Prepare artifacts: ready


## 1. Write the materialization script

In [2]:
materialize_script_text = '#!/usr/bin/env python3\n"""Materialize OmniLingual 300M Levantine full data from the Whisper pipeline split logic.\n\nThis script rebuilds the train/validation/test split from the raw Casablanca parquet\nand Omnilingual Arrow files used by the Whisper Medium Levantine pipeline.\n\nImportant difference from the Whisper script:\n- This does NOT drop clips >= 30 seconds.\n- It only drops empty-text rows and rows below min_audio_seconds.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nimport json\nimport os\nimport re\nimport shutil\nimport sys\nimport unicodedata\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport soundfile as sf\n\n\nREPO_ROOT = Path("/home/MohammadNabulsi/whisper")\nBUNDLE_DIR = Path(__file__).resolve().parent\n\nDATASET_NAME = "omnilingual_300m_levantine_full_from_whisper_splits"\nOUTPUT_DATASET_DIR = BUNDLE_DIR / "datasets" / DATASET_NAME\nOUTPUT_AUDIO_DIR = OUTPUT_DATASET_DIR / "audio"\n\nSAMPLE_RATE = 16_000\nMIN_AUDIO_SECONDS = 0.3\nSPLIT_SEED = 42\nFORCE_REBUILD = os.environ.get("FORCE_REBUILD_DATASET", "").strip().lower() in {"1", "true", "yes", "y"}\n\nEXPECTED_OUTPUT_FILES = (\n    OUTPUT_DATASET_DIR / "dataset_summary.json",\n    OUTPUT_DATASET_DIR / "train.jsonl",\n    OUTPUT_DATASET_DIR / "validation.jsonl",\n    OUTPUT_DATASET_DIR / "test.jsonl",\n)\n\nTRAIN_PARQUET_FILES = (\n    REPO_ROOT / "casablanca" / "levant" / "Palestine" / "validation-00001-of-00002.parquet",\n    REPO_ROOT / "casablanca" / "levant" / "Palestine" / "validation-00000-of-00002.parquet",\n    REPO_ROOT / "casablanca" / "levant" / "Jordan" / "validation-00000-of-00001.parquet",\n)\n\nEVAL_PARQUET_FILES = (\n    REPO_ROOT / "casablanca" / "levant" / "Palestine" / "test-00001-of-00002.parquet",\n    REPO_ROOT / "casablanca" / "levant" / "Palestine" / "test-00000-of-00002.parquet",\n    REPO_ROOT / "casablanca" / "levant" / "Jordan" / "test-00000-of-00001.parquet",\n)\n\nTRAIN_ARROW_FILES = (\n    REPO_ROOT / "omnilingual_selected" / "apc_north_levantine_all_splits" / "data-00001-of-00003.arrow",\n    REPO_ROOT / "omnilingual_selected" / "apc_north_levantine_all_splits" / "data-00000-of-00003.arrow",\n)\n\nEVAL_ARROW_FILES = (\n    REPO_ROOT / "omnilingual_selected" / "apc_north_levantine_all_splits" / "data-00002-of-00003.arrow",\n)\n\n\nAR_DIACRITICS_RE = re.compile(r"[\\u0610-\\u061a\\u064b-\\u065f\\u0670\\u06d6-\\u06ed]")\nCONTROL_RE = re.compile(r"[\\u200b-\\u200f\\u202a-\\u202e\\u2066-\\u2069\\ufeff]")\nSPACE_RE = re.compile(r"\\s+")\nASR_TAG_RE = re.compile(r"<[^>]+>|\\[[^\\]]+\\]")\nAR_PUNCT_SPACING_RE = re.compile(r"\\s*([،؛؟,.!?;:])\\s*")\nREPEATED_PUNCT_RE = re.compile(r"([،؛؟,.!?;:])\\1+")\n\n\ndef normalize_arabic_text(text: Any) -> str:\n    if text is None:\n        return ""\n    text = unicodedata.normalize("NFKC", str(text))\n    text = CONTROL_RE.sub(" ", text)\n    text = ASR_TAG_RE.sub(" ", text)\n    text = text.replace("ـ", "")\n    text = AR_DIACRITICS_RE.sub("", text)\n    text = AR_PUNCT_SPACING_RE.sub(r" \\1 ", text)\n    text = REPEATED_PUNCT_RE.sub(r"\\1", text)\n    text = SPACE_RE.sub(" ", text).strip()\n    return text\n\n\ndef jsonl_write(path: Path, rows: list[dict[str, Any]]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for row in rows:\n            handle.write(json.dumps(row, ensure_ascii=False) + "\\n")\n\n\ndef safe_name(value: str) -> str:\n    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", value)\n    return cleaned.strip("._") or "sample"\n\n\ndef stable_hash(text: str, seed: int) -> int:\n    payload = f"{seed}|{text}".encode("utf-8")\n    return int(hashlib.sha256(payload).hexdigest(), 16)\n\n\ndef row_uid(prefix: str, path: Path, row_idx: int) -> str:\n    digest = hashlib.sha1(f"{prefix}:{path}:{row_idx}".encode("utf-8")).hexdigest()\n    return f"{prefix}-{digest[:16]}"\n\n\ndef stable_row_order(rows: list[dict[str, Any]], *, seed: int, salt: str) -> list[dict[str, Any]]:\n    return sorted(rows, key=lambda row: stable_hash(f"{salt}|{row.get(\'uid\')}", seed))\n\n\ndef split_rows_half(rows: list[dict[str, Any]], *, seed: int, salt: str) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:\n    ordered = stable_row_order(rows, seed=seed, salt=salt)\n    midpoint = len(ordered) // 2\n    val_rows = [dict(row, split="validation") for row in ordered[:midpoint]]\n    test_rows = [dict(row, split="test") for row in ordered[midpoint:]]\n    return val_rows, test_rows\n\n\ndef ensure_audio_tools():\n    try:\n        import librosa\n    except ImportError:\n        librosa = None\n    try:\n        import pyarrow as pa\n        import pyarrow.ipc as pa_ipc\n        import pyarrow.parquet as pq\n    except ImportError as exc:\n        raise RuntimeError("pyarrow is required for parquet/arrow reading.") from exc\n    return librosa, pa, pa_ipc, pq\n\n\ndef to_mono_float32(audio: Any) -> np.ndarray:\n    arr = np.asarray(audio, dtype=np.float32)\n    if arr.ndim == 0:\n        return arr.reshape(1)\n    if arr.ndim == 1:\n        return arr\n    if arr.ndim == 2:\n        if arr.shape[0] < arr.shape[1]:\n            return arr.mean(axis=0).astype(np.float32)\n        return arr.mean(axis=1).astype(np.float32)\n    return arr.reshape(-1).astype(np.float32)\n\n\ndef read_audio_bytes(payload: bytes) -> tuple[np.ndarray, int]:\n    audio, sample_rate = sf.read(io.BytesIO(payload), dtype="float32", always_2d=False)\n    return to_mono_float32(audio), int(sample_rate)\n\n\ndef read_audio_path(path: str) -> tuple[np.ndarray, int]:\n    audio, sample_rate = sf.read(path, dtype="float32", always_2d=False)\n    return to_mono_float32(audio), int(sample_rate)\n\n\ndef resample_audio(audio: np.ndarray, sample_rate: int, target_sample_rate: int) -> np.ndarray:\n    if int(sample_rate) == int(target_sample_rate):\n        return to_mono_float32(audio)\n    librosa, *_ = ensure_audio_tools()\n    if librosa is None:\n        raise RuntimeError("librosa is required when an input sample rate is not already 16 kHz.")\n    return librosa.resample(\n        to_mono_float32(audio),\n        orig_sr=int(sample_rate),\n        target_sr=int(target_sample_rate),\n    ).astype(np.float32)\n\n\ndef read_parquet_row(path: str, row_idx: int) -> dict[str, Any]:\n    *_, pq = ensure_audio_tools()\n    parquet_file = pq.ParquetFile(path)\n    seen = 0\n    for batch in parquet_file.iter_batches(batch_size=1024):\n        py_rows = batch.to_pylist()\n        if seen + len(py_rows) > row_idx:\n            return py_rows[row_idx - seen]\n        seen += len(py_rows)\n    raise IndexError(f"row_idx={row_idx} out of range for {path}")\n\n\ndef read_arrow_row(path: str, row_idx: int, columns: list[str] | None = None) -> dict[str, Any]:\n    _, pa, pa_ipc, _ = ensure_audio_tools()\n    index = int(row_idx)\n    seen = 0\n    with pa.memory_map(path, "r") as source:\n        reader = pa_ipc.open_stream(source)\n        for batch in reader:\n            if seen + batch.num_rows <= index:\n                seen += batch.num_rows\n                continue\n            local = index - seen\n            names = set(batch.schema.names)\n            use_cols = [column for column in (columns or batch.schema.names) if column in names]\n            return {\n                column: batch.column(batch.schema.get_field_index(column)).to_pylist()[local]\n                for column in use_cols\n            }\n    raise IndexError(f"row_idx={row_idx} out of range for {path}")\n\n\ndef load_parquet_audio(row: dict[str, Any]) -> tuple[np.ndarray, int]:\n    example = read_parquet_row(str(row["parquet_file"]), int(row["row_idx"]))\n    audio = example.get("audio") or {}\n    if isinstance(audio, dict):\n        if audio.get("bytes") is not None:\n            return read_audio_bytes(audio["bytes"])\n        if audio.get("path"):\n            candidate = str(audio["path"])\n            if not Path(candidate).is_absolute():\n                candidate = str(Path(str(row["parquet_file"])).parent / candidate)\n            return read_audio_path(candidate)\n    raise RuntimeError(f"Unsupported Parquet audio payload for uid={row.get(\'uid\')}")\n\n\ndef load_arrow_audio(row: dict[str, Any]) -> tuple[np.ndarray, int]:\n    example = read_arrow_row(str(row["arrow_file"]), int(row["row_idx"]), columns=["audio"])\n    audio = example.get("audio")\n    if isinstance(audio, dict):\n        if audio.get("array") is not None:\n            sample_rate = audio.get("sampling_rate") or SAMPLE_RATE\n            return to_mono_float32(audio["array"]), int(sample_rate)\n        if audio.get("bytes") is not None:\n            return read_audio_bytes(audio["bytes"])\n        if audio.get("path"):\n            candidate = str(audio["path"])\n            if not Path(candidate).is_absolute():\n                candidate = str(Path(str(row["arrow_file"])).parent / candidate)\n            return read_audio_path(candidate)\n    raise RuntimeError(f"Unsupported Arrow audio payload for uid={row.get(\'uid\')}")\n\n\ndef load_audio_for_manifest_row(row: dict[str, Any]) -> tuple[np.ndarray, int]:\n    if row.get("audio_kind") == "hf_audio":\n        audio, sample_rate = load_arrow_audio(row)\n    elif row.get("audio_kind") == "bytes":\n        audio, sample_rate = load_parquet_audio(row)\n    elif row.get("audio_kind") == "path" and row.get("audio_path"):\n        audio, sample_rate = read_audio_path(str(row["audio_path"]))\n    else:\n        raise RuntimeError(f"Unsupported audio_kind={row.get(\'audio_kind\')} uid={row.get(\'uid\')}")\n    return resample_audio(audio, int(sample_rate), SAMPLE_RATE), SAMPLE_RATE\n\n\ndef build_parquet_rows(paths: tuple[Path, ...], *, original_split: str, source_group: str, source_label: str) -> list[dict[str, Any]]:\n    *_, pq = ensure_audio_tools()\n    rows: list[dict[str, Any]] = []\n    for path in paths:\n        parquet_file = pq.ParquetFile(path)\n        row_idx = 0\n        for batch in parquet_file.iter_batches(\n            batch_size=1024,\n            columns=["seg_id", "transcription", "gender", "duration"],\n        ):\n            for item in batch.to_pylist():\n                rows.append(\n                    {\n                        "uid": row_uid("parquet", path, row_idx),\n                        "source": "casablanca",\n                        "source_group": source_group,\n                        "source_root": str(path.parent),\n                        "original_split": original_split,\n                        "parquet_file": str(path),\n                        "arrow_file": None,\n                        "row_idx": row_idx,\n                        "audio_kind": "bytes",\n                        "text": item.get("transcription", ""),\n                        "duration": item.get("duration"),\n                        "segment_id": item.get("seg_id"),\n                        "speaker_id": None,\n                        "gender": item.get("gender"),\n                        "language": "apc_Arab",\n                        "metadata": {\n                            "source_label": source_label,\n                            "country": path.parent.name,\n                            "filename": path.name,\n                        },\n                    }\n                )\n                row_idx += 1\n    return rows\n\n\ndef build_arrow_rows(paths: tuple[Path, ...], *, original_split: str, source_group: str) -> list[dict[str, Any]]:\n    _, pa, pa_ipc, _ = ensure_audio_tools()\n    rows: list[dict[str, Any]] = []\n    columns = [\n        "language",\n        "speaker_id",\n        "prompt_id",\n        "prompt",\n        "segment_id",\n        "duration",\n        "raw_text",\n        "iso_639_3",\n        "glottocode",\n        "iso_15924",\n        "config",\n        "original_split",\n    ]\n    for path in paths:\n        row_idx = 0\n        with pa.memory_map(str(path), "r") as source:\n            reader = pa_ipc.open_stream(source)\n            for batch in reader:\n                names = set(batch.schema.names)\n                use_cols = [column for column in columns if column in names]\n                payload = {\n                    column: batch.column(batch.schema.get_field_index(column)).to_pylist()\n                    for column in use_cols\n                }\n                for offset in range(batch.num_rows):\n                    def value(column: str, default: Any = None) -> Any:\n                        return payload.get(column, [default] * batch.num_rows)[offset]\n\n                    metadata = {\n                        "config": value("config"),\n                        "glottocode": value("glottocode"),\n                        "iso_639_3": value("iso_639_3"),\n                        "iso_15924": value("iso_15924"),\n                        "prompt": value("prompt"),\n                        "prompt_id": value("prompt_id"),\n                        "filename": path.name,\n                    }\n                    rows.append(\n                        {\n                            "uid": row_uid("arrow", path, row_idx),\n                            "source": "omnilingual",\n                            "source_group": source_group,\n                            "source_root": str(path.parent),\n                            "original_split": value("original_split", original_split) or original_split,\n                            "parquet_file": None,\n                            "arrow_file": str(path),\n                            "row_idx": row_idx,\n                            "audio_kind": "hf_audio",\n                            "text": value("raw_text", ""),\n                            "duration": value("duration"),\n                            "segment_id": value("segment_id"),\n                            "speaker_id": value("speaker_id"),\n                            "gender": None,\n                            "language": value("language", "apc_Arab"),\n                            "metadata": metadata,\n                        }\n                    )\n                    row_idx += 1\n    return rows\n\n\ndef filter_rows(rows: list[dict[str, Any]], split: str) -> tuple[list[dict[str, Any]], dict[str, int]]:\n    kept: list[dict[str, Any]] = []\n    stats = {"input": len(rows), "kept": 0, "dropped_empty": 0, "dropped_too_short": 0, "dropped_bad_duration": 0}\n    for row in rows:\n        text = normalize_arabic_text(row.get("text", ""))\n        if not text:\n            stats["dropped_empty"] += 1\n            continue\n        try:\n            duration = float(row.get("duration"))\n        except Exception:\n            stats["dropped_bad_duration"] += 1\n            continue\n        if duration < MIN_AUDIO_SECONDS:\n            stats["dropped_too_short"] += 1\n            continue\n\n        clean = dict(row)\n        clean["text"] = text\n        clean["duration"] = duration\n        clean["split"] = split\n        kept.append(clean)\n\n    stats["kept"] = len(kept)\n    return kept, stats\n\n\ndef rows_by_source_group(rows: list[dict[str, Any]]) -> dict[str, dict[str, float]]:\n    out: dict[str, dict[str, float]] = {}\n    for group in sorted({str(row.get("source_group")) for row in rows}):\n        group_rows = [row for row in rows if str(row.get("source_group")) == group]\n        out[group] = {\n            "rows": len(group_rows),\n            "hours": sum(float(row.get("duration") or 0.0) for row in group_rows) / 3600.0,\n        }\n    return out\n\n\ndef materialize_rows(rows: list[dict[str, Any]], split: str) -> tuple[list[dict[str, Any]], dict[str, Any]]:\n    split_audio_dir = OUTPUT_AUDIO_DIR / split\n    split_audio_dir.mkdir(parents=True, exist_ok=True)\n\n    output_rows: list[dict[str, Any]] = []\n    source_group_counts: Counter[str] = Counter()\n    source_hours: Counter[str] = Counter()\n\n    for idx, row in enumerate(rows):\n        audio, sample_rate = load_audio_for_manifest_row(row)\n        uid = str(row.get("uid") or f"{split}_{idx:06d}")\n        audio_name = f"{idx:06d}_{safe_name(uid)}.flac"\n        audio_path = split_audio_dir / audio_name\n        sf.write(audio_path, audio, sample_rate, format="FLAC")\n\n        duration = float(row.get("duration") or (len(audio) / float(sample_rate)))\n        source_group = str(row.get("source_group") or row.get("source") or "unknown")\n        source_group_counts[source_group] += 1\n        source_hours[source_group] += duration / 3600.0\n\n        output_rows.append(\n            {\n                "uid": uid,\n                "audio_path": str(audio_path),\n                "text": str(row.get("text") or "").strip(),\n                "duration": duration,\n                "split": split,\n                "language": str(row.get("language") or "ar"),\n                "speaker_id": str(row.get("speaker_id") or ""),\n                "source": row.get("source"),\n                "source_group": source_group,\n                "segment_id": row.get("segment_id"),\n                "gender": row.get("gender"),\n                "metadata": {\n                    **(row.get("metadata") or {}),\n                    "materialized_from": {\n                        "parquet_file": row.get("parquet_file"),\n                        "arrow_file": row.get("arrow_file"),\n                        "row_idx": row.get("row_idx"),\n                        "source_root": row.get("source_root"),\n                        "original_split": row.get("original_split"),\n                        "audio_kind": row.get("audio_kind"),\n                    },\n                },\n            }\n        )\n\n        if (idx + 1) % 100 == 0:\n            print(f"[{split}] materialized {idx + 1}/{len(rows)}", flush=True)\n\n    summary = {\n        "count": len(output_rows),\n        "hours": sum(source_hours.values()),\n        "source_group_counts": dict(source_group_counts),\n        "source_group_hours": {key: round(value, 6) for key, value in source_hours.items()},\n    }\n    return output_rows, summary\n\n\n\ndef dataset_outputs_ready() -> bool:\n    return all(path.exists() for path in EXPECTED_OUTPUT_FILES)\n\n\ndef check_inputs() -> None:\n    all_paths = list(TRAIN_PARQUET_FILES) + list(EVAL_PARQUET_FILES) + list(TRAIN_ARROW_FILES) + list(EVAL_ARROW_FILES)\n    missing = [str(path) for path in all_paths if not path.exists()]\n    if missing:\n        raise FileNotFoundError("Missing input files:\\n" + "\\n".join(missing))\n\n\ndef main() -> None:\n    check_inputs()\n\n    outputs_ready = dataset_outputs_ready()\n    if outputs_ready and not FORCE_REBUILD:\n        print(\n            f"Dataset already materialized at {OUTPUT_DATASET_DIR}; skipping rebuild. "\n            "Set FORCE_REBUILD_DATASET=1 to rebuild.",\n            flush=True,\n        )\n        return\n\n    if OUTPUT_DATASET_DIR.exists() and (FORCE_REBUILD or not outputs_ready):\n        reason = "force rebuild requested" if FORCE_REBUILD else "existing outputs are incomplete"\n        print(f"Removing previous output dataset ({reason}): {OUTPUT_DATASET_DIR}", flush=True)\n        shutil.rmtree(OUTPUT_DATASET_DIR)\n\n    OUTPUT_DATASET_DIR.mkdir(parents=True, exist_ok=True)\n\n    print("Building raw row references from Whisper split sources...", flush=True)\n\n    train_rows_raw = build_parquet_rows(\n        TRAIN_PARQUET_FILES,\n        original_split="validation",\n        source_group="casablanca_levantine_train",\n        source_label="casablanca_levantine_validation_as_train",\n    ) + build_arrow_rows(\n        TRAIN_ARROW_FILES,\n        original_split="train",\n        source_group="omnilingual_apc_north_levantine_train",\n    )\n\n    eval_parquet_raw = build_parquet_rows(\n        EVAL_PARQUET_FILES,\n        original_split="test",\n        source_group="casablanca_levantine_eval_pool",\n        source_label="casablanca_levantine_test_split_50_50",\n    )\n\n    eval_arrow_raw = build_arrow_rows(\n        EVAL_ARROW_FILES,\n        original_split="train",\n        source_group="omnilingual_apc_north_levantine_eval_pool",\n    )\n\n    filtered_train, train_filter_stats = filter_rows(train_rows_raw, "train")\n    filtered_eval_parquet, eval_parquet_filter_stats = filter_rows(eval_parquet_raw, "eval_pool")\n    filtered_eval_arrow, eval_arrow_filter_stats = filter_rows(eval_arrow_raw, "eval_pool")\n\n    parquet_val_rows, parquet_test_rows = split_rows_half(\n        filtered_eval_parquet,\n        seed=SPLIT_SEED,\n        salt="custom-casablanca-heldout",\n    )\n    arrow_val_rows, arrow_test_rows = split_rows_half(\n        filtered_eval_arrow,\n        seed=SPLIT_SEED,\n        salt="custom-omnilingual-heldout",\n    )\n\n    train_rows = [dict(row, split="train") for row in stable_row_order(filtered_train, seed=SPLIT_SEED, salt="custom-train")]\n    validation_rows = stable_row_order(parquet_val_rows + arrow_val_rows, seed=SPLIT_SEED, salt="custom-val")\n    test_rows = stable_row_order(parquet_test_rows + arrow_test_rows, seed=SPLIT_SEED, salt="custom-test")\n\n    print(f"Selected rows: train={len(train_rows)} validation={len(validation_rows)} test={len(test_rows)}", flush=True)\n    print("Materializing audio to FLAC...", flush=True)\n\n    output_rows: dict[str, list[dict[str, Any]]] = {}\n    split_summary: dict[str, dict[str, Any]] = {}\n\n    for split, rows in [("train", train_rows), ("validation", validation_rows), ("test", test_rows)]:\n        rows_out, summary = materialize_rows(rows, split)\n        output_rows[split] = rows_out\n        split_summary[split] = summary\n        jsonl_write(OUTPUT_DATASET_DIR / f"{split}.jsonl", rows_out)\n\n    combined_rows = output_rows["train"] + output_rows["validation"] + output_rows["test"]\n    jsonl_write(OUTPUT_DATASET_DIR / "combined.jsonl", combined_rows)\n\n    selection_summary = {\n        "dataset_name": DATASET_NAME,\n        "dataset_dir": str(OUTPUT_DATASET_DIR),\n        "sample_rate": SAMPLE_RATE,\n        "min_audio_seconds": MIN_AUDIO_SECONDS,\n        "drop_audio_at_or_above_seconds": None,\n        "note_on_long_audio": "No >=30s rule is applied here. Long clips are kept during materialization.",\n        "raw_counts": {\n            "train_rows_raw": len(train_rows_raw),\n            "eval_parquet_raw": len(eval_parquet_raw),\n            "eval_arrow_raw": len(eval_arrow_raw),\n        },\n        "filter_stats": {\n            "train": train_filter_stats,\n            "eval_parquet_pool": eval_parquet_filter_stats,\n            "eval_arrow_pool": eval_arrow_filter_stats,\n        },\n        "selected_counts": {split: len(rows) for split, rows in output_rows.items()},\n        "selected_hours": {\n            split: round(split_summary[split]["hours"], 6)\n            for split in ["train", "validation", "test"]\n        },\n        "source_group_breakdown": split_summary,\n        "train_by_source_group": rows_by_source_group(train_rows),\n        "validation_by_source_group": rows_by_source_group(validation_rows),\n        "test_by_source_group": rows_by_source_group(test_rows),\n        "data_sources": {\n            "train_parquet_files": [str(path) for path in TRAIN_PARQUET_FILES],\n            "eval_parquet_files": [str(path) for path in EVAL_PARQUET_FILES],\n            "train_arrow_files": [str(path) for path in TRAIN_ARROW_FILES],\n            "eval_arrow_files": [str(path) for path in EVAL_ARROW_FILES],\n        },\n        "output_files": {\n            "train": str(OUTPUT_DATASET_DIR / "train.jsonl"),\n            "validation": str(OUTPUT_DATASET_DIR / "validation.jsonl"),\n            "test": str(OUTPUT_DATASET_DIR / "test.jsonl"),\n            "combined": str(OUTPUT_DATASET_DIR / "combined.jsonl"),\n        },\n        "split_logic": {\n            "train": [\n                "Casablanca validation Palestine/Jordan parquet files are used as train.",\n                "Omnilingual APC Arrow shards data-00001 and data-00000 are used as train.",\n            ],\n            "validation": [\n                "Half of Casablanca test Palestine/Jordan parquet pool using stable hash split.",\n                "Half of Omnilingual APC Arrow data-00002 using stable hash split.",\n            ],\n            "test": [\n                "Other half of Casablanca test Palestine/Jordan parquet pool using stable hash split.",\n                "Other half of Omnilingual APC Arrow data-00002 using stable hash split.",\n            ],\n        },\n    }\n\n    summary_path = OUTPUT_DATASET_DIR / "dataset_summary.json"\n    summary_path.write_text(json.dumps(selection_summary, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n\n    print(json.dumps(selection_summary, ensure_ascii=False, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'

write_text_if_changed(MATERIALIZE_SCRIPT, materialize_script_text, encoding="utf-8")
print("Size:", MATERIALIZE_SCRIPT.stat().st_size, "bytes")


Unchanged: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/prepare_omnilingual_300m_levantine_full_from_whisper_splits.py
Size: 24198 bytes


## 2. Syntax-check required scripts

In [3]:

assert MATERIALIZE_SCRIPT.exists(), f"Missing materializer: {MATERIALIZE_SCRIPT}"
assert TRAINER_SCRIPT.exists(), f"Missing trainer: {TRAINER_SCRIPT}"

for path in [MATERIALIZE_SCRIPT, TRAINER_SCRIPT]:
    print("Checking:", path)
    py_compile.compile(str(path), doraise=True)
    print("OK")


Checking: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/prepare_omnilingual_300m_levantine_full_from_whisper_splits.py
OK
Checking: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/asr_universal_trainer.py
OK



## 3. Materialize full data

This uses the Whisper pipeline data sources:

- Casablanca validation Palestine/Jordan as train
- Casablanca test Palestine/Jordan split 50/50 into validation/test
- Omnilingual APC Arrow shards `data-00001` + `data-00000` as train
- Omnilingual APC Arrow shard `data-00002` split 50/50 into validation/test

Long audio is kept. The script only drops empty text and clips shorter than `0.3s`.


In [4]:
if outputs_exist(DATASET_REQUIRED_FILES) and not FORCE_MATERIALIZE:
    print("Dataset artifacts already exist; skipping materialization.")
    print("Set FORCE_MATERIALIZE = True in the first cell to rebuild them.")
else:
    missing = missing_outputs(DATASET_REQUIRED_FILES)
    if missing:
        print("Materialization required; missing outputs:")
        for path in missing:
            print(" -", path)
    env = dict(os.environ)
    if FORCE_MATERIALIZE:
        env["FORCE_REBUILD_DATASET"] = "1"
    cmd = [sys.executable, str(MATERIALIZE_SCRIPT)]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, env=env)
    assert result.returncode == 0, result.returncode

describe_outputs("Dataset artifacts", DATASET_REQUIRED_FILES)


Running: /home/MohammadNabulsi/whisper/.venv/bin/python /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/prepare_omnilingual_300m_levantine_full_from_whisper_splits.py
Removing previous output dataset (force rebuild requested): /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/datasets/omnilingual_300m_levantine_full_from_whisper_splits
Building raw row references from Whisper split sources...
Selected rows: train=1859 validation=843 test=843
Materializing audio to FLAC...
[train] materialized 100/1859
[train] materialized 200/1859
[train] materialized 300/1859
[train] materialized 400/1859
[train] materialized 500/1859
[train] materialized 600/1859
[train] materialized 700/1859
[train] materialized 800/1859
[train] materialized 900/1859
[train] materialized 1000/1859
[train] materialized 1100/1859
[train] materialized 1200/1859
[train] materialized 1300/1859
[train] materialized 1400/1859
[train] materialized 1500/1859
[train] materialized 1600/1859
[train] materialized

## 4. Inspect materialized dataset

In [5]:

for rel in ["dataset_summary.json", "train.jsonl", "validation.jsonl", "test.jsonl"]:
    path = DATASET_DIR / rel
    print(f"\n--- {path} ---")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:5000])
    else:
        print("missing")



--- /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/datasets/omnilingual_300m_levantine_full_from_whisper_splits/dataset_summary.json ---
{
  "dataset_name": "omnilingual_300m_levantine_full_from_whisper_splits",
  "dataset_dir": "/home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/datasets/omnilingual_300m_levantine_full_from_whisper_splits",
  "sample_rate": 16000,
  "min_audio_seconds": 0.3,
  "drop_audio_at_or_above_seconds": null,
  "note_on_long_audio": "No >=30s rule is applied here. Long clips are kept during materialization.",
  "raw_counts": {
    "train_rows_raw": 1860,
    "eval_parquet_raw": 1515,
    "eval_arrow_raw": 172
  },
  "filter_stats": {
    "train": {
      "input": 1860,
      "kept": 1859,
      "dropped_empty": 0,
      "dropped_too_short": 1,
      "dropped_bad_duration": 0
    },
    "eval_parquet_pool": {
      "input": 1515,
      "kept": 1514,
      "dropped_empty": 0,
      "dropped_too_short": 1,
      "dropped_bad_duration": 0
    },
  

## 5. Create the matching non-smoke config

In [6]:
try:
    import yaml
except ImportError as exc:
    raise RuntimeError("Install pyyaml first: pip install pyyaml") from exc

if BASE_CONFIG_PATH.exists():
    cfg = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8")) or {}
    print("Loaded base config:", BASE_CONFIG_PATH)
else:
    print("Base config missing; creating a minimal config.")
    cfg = {}

cfg.setdefault("run", {})
cfg.setdefault("model", {})
cfg.setdefault("data", {})
cfg.setdefault("dataset", {})
cfg.setdefault("training", {})
cfg.setdefault("eval", {})
cfg.setdefault("output", {})

# Run/model identity.
cfg["run"]["name"] = DATASET_NAME
cfg["run"]["smoke"] = False
cfg["run"]["is_smoke"] = False
cfg["model"]["model_id"] = "facebook/omniASR-LLM-300M"
cfg["model"].setdefault("family", "omni_llm")

# Output locations.
cfg["output"]["work_dir"] = str(WORK_DIR)
cfg["output"]["output_dir"] = str(WORK_DIR)
cfg["output_dir"] = str(WORK_DIR)

# Dataset paths. Several aliases are filled because different trainer versions use different key names.
split_paths = {
    "train": str(DATASET_DIR / "train.jsonl"),
    "validation": str(DATASET_DIR / "validation.jsonl"),
    "test": str(DATASET_DIR / "test.jsonl"),
}
for block_name in ["data", "dataset"]:
    block = cfg.setdefault(block_name, {})
    block.pop("dataset_registry_path", None)
    block["format"] = "jsonl"
    block["name"] = DATASET_NAME
    block["dataset_name"] = DATASET_NAME
    block["data_dir"] = str(DATASET_DIR)
    block["train_manifest"] = str(DATASET_DIR / "train.jsonl")
    block["validation_manifest"] = str(DATASET_DIR / "validation.jsonl")
    block["val_manifest"] = str(DATASET_DIR / "validation.jsonl")
    block["eval_manifest"] = str(DATASET_DIR / "validation.jsonl")
    block["test_manifest"] = str(DATASET_DIR / "test.jsonl")
    block["train_path"] = str(DATASET_DIR / "train.jsonl")
    block["validation_path"] = str(DATASET_DIR / "validation.jsonl")
    block["val_path"] = str(DATASET_DIR / "validation.jsonl")
    block["test_path"] = str(DATASET_DIR / "test.jsonl")
    block["audio_column"] = "audio_path"
    block["text_column"] = "text"
    block["transcript_column"] = "text"
    block["duration_column"] = "duration"
    block["split_column"] = "split"
    block["long_audio_policy"] = "keep"
    block["drop_audio_at_or_above_seconds"] = None
    block["max_audio_seconds"] = None
    block["split_paths"] = dict(split_paths)

cfg["dataset"]["splits"] = {
    **split_paths,
    "val": split_paths["validation"],
}
cfg["data"]["splits"] = cfg["dataset"]["splits"]


# Remove any stale smoke/sample caps inherited from the base config.
# This prevents accidentally preparing/training on only 1 sample from an old smoke config.
SAMPLE_CAP_KEYS = [
    "max_train_samples", "max_eval_samples", "max_validation_samples", "max_test_samples",
    "train_sample_cap", "eval_sample_cap", "validation_sample_cap", "test_sample_cap",
    "sample_cap", "limit", "max_samples", "num_samples", "subset_size",
    "train_limit", "eval_limit", "validation_limit", "test_limit",
    "smoke_train_samples", "smoke_eval_samples", "smoke_test_samples",
]
for block_name in ["run", "data", "dataset", "training", "eval"]:
    block = cfg.setdefault(block_name, {})
    removed = {}
    for key in SAMPLE_CAP_KEYS:
        if key in block:
            removed[key] = block.pop(key)
    if removed:
        print(f"Removed sample-cap keys from {block_name}:", removed)

# Keep intended full run settings if present, but ensure non-smoke and long-audio keep.
cfg["training"]["num_train_epochs"] = cfg["training"].get("num_train_epochs", 20)
cfg["training"]["long_audio_policy"] = "keep"
cfg["training"]["drop_audio_at_or_above_seconds"] = None
cfg["training"]["max_audio_seconds"] = None

config_text = yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False)
write_text_if_changed(CONFIG_PATH, config_text, encoding="utf-8")

print("Wrote:", CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding="utf-8")[:5000])


Loaded base config: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/omnilingual_300m_levantine_full.yaml
Wrote: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/omnilingual_300m_levantine_full_from_whisper_splits.yaml
Wrote: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/omnilingual_300m_levantine_full_from_whisper_splits.yaml
model:
  model_id: facebook/omniASR-LLM-300M
  sample_rate: 16000
  prompt_template: Transcribe this audio in Arabic.
  spec_overrides:
    backend: omni_external
    family: omni_llm
    max_train_seconds: null
    max_eval_chunk_seconds: null
  family: omni_llm
data:
  dataset_name: omnilingual_300m_levantine_full_from_whisper_splits
  language: ar
  long_audio_policy: keep
  format: jsonl
  name: omnilingual_300m_levantine_full_from_whisper_splits
  data_dir: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/datasets/omnilingual_300m_levantine_full_from_whisper_splits
  train_manifest: /home/MohammadNabulsi/whisper/Runs/asr_pl

## 6. Run trainer prepare/export

In [7]:
def run_prepare(force: bool = False):
    if outputs_exist(PREPARE_REQUIRED_FILES) and not force:
        print("Prepare artifacts already exist; skipping trainer prepare/export.")
        print("Set FORCE_PREPARE = True in the first cell to rerun it.")
        return ["skipped", "prepare outputs already exist"]

    prepare_dir = WORK_DIR / "prepared"
    if prepare_dir.exists() and (force or missing_outputs(PREPARE_REQUIRED_FILES)):
        reason = "force rerun requested" if force else "outputs are incomplete"
        print(f"Removing partial prepare directory ({reason}): {prepare_dir}")
        shutil.rmtree(prepare_dir)

    candidates = [
        [sys.executable, str(TRAINER_SCRIPT), "--config", str(CONFIG_PATH), "--stage", "prepare"],
        [sys.executable, str(TRAINER_SCRIPT), "--config", str(CONFIG_PATH), "--phase", "prepare"],
        [sys.executable, str(TRAINER_SCRIPT), "prepare", "--config", str(CONFIG_PATH)],
    ]

    help_cmd = [sys.executable, str(TRAINER_SCRIPT), "--help"]
    print("\nTrainer help:")
    help_result = subprocess.run(help_cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
    print(help_result.stdout)
    if help_result.stderr:
        print(help_result.stderr)

    last = None
    for cmd in candidates:
        print("\nTrying:", " ".join(cmd))
        result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode == 0:
            print("Prepare/export succeeded with:", " ".join(cmd))
            return cmd
        last = result

    raise RuntimeError(f"All prepare commands failed. Last exit code={last.returncode}")

successful_prepare_cmd = run_prepare(force=FORCE_PREPARE)
describe_outputs("Prepare artifacts", PREPARE_REQUIRED_FILES)


Removing partial prepare directory (force rerun requested): /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/runs/omnilingual_300m_levantine_full_from_whisper_splits/prepared

Trainer help:
usage: asr_universal_trainer.py [-h] [--config CONFIG]
                                [--stage {prepare,train,eval,all}]
                                [--smoke-test] [--work-dir WORK_DIR]

Universal ASR plug-and-play trainer/evaluator

options:
  -h, --help            show this help message and exit
  --config CONFIG       YAML/JSON config path
  --stage {prepare,train,eval,all}
                        Pipeline stage
  --smoke-test          Create a 1 train / 1 val / 1 test mock run and execute
                        it
  --work-dir WORK_DIR   Work dir for smoke-test if --config omitted


Trying: /home/MohammadNabulsi/whisper/.venv/bin/python /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/asr_universal_trainer.py --config /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer

## 7. Validate this is not a stale 1-sample smoke prepare


In [8]:

# Validate manifest row counts so a stale smoke/1-sample prepare cannot silently pass.
def count_jsonl(path: Path) -> int:
    if not path.exists():
        return -1
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

manifest_pairs = [
    ("dataset train", DATASET_DIR / "train.jsonl"),
    ("dataset validation", DATASET_DIR / "validation.jsonl"),
    ("dataset test", DATASET_DIR / "test.jsonl"),
    ("prepared train", WORK_DIR / "prepared" / "train.jsonl"),
    ("prepared validation", WORK_DIR / "prepared" / "validation.jsonl"),
    ("prepared test", WORK_DIR / "prepared" / "test.jsonl"),
]

counts = {name: count_jsonl(path) for name, path in manifest_pairs}
print(json.dumps(counts, indent=2, ensure_ascii=False))

assert counts["dataset train"] > 1, "Dataset train manifest has <=1 row; materialization did not produce a full dataset."
assert counts["prepared train"] > 1, "Prepared train manifest has <=1 row; stale smoke output is still being used."
assert counts["prepared train"] == counts["dataset train"], (
    "Prepared train row count does not match dataset train row count. "
    "Check trainer filtering/logs before starting training."
)
print("Full-run manifest validation passed.")


{
  "dataset train": 1859,
  "dataset validation": 843,
  "dataset test": 843,
  "prepared train": 1859,
  "prepared validation": 843,
  "prepared test": 843
}
Full-run manifest validation passed.


## 7. Inspect outputs

In [9]:

for rel in [
    "run.log",
    "run_result.json",
    "prepared/stats.json",
    "prepared/dataset_resolved.json",
    "prepared/model_spec.json",
]:
    path = WORK_DIR / rel
    print(f"\n--- {path} ---")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:5000])
    else:
        print("missing")



--- /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/runs/omnilingual_300m_levantine_full_from_whisper_splits/run.log ---
2026-06-26 16:35:59,409 | INFO | Logging to /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/runs/omnilingual_300m_levantine_full_from_whisper_splits/run.log
2026-06-26 16:35:59,409 | INFO | Model spec: {'model_id': 'facebook/omniASR-LLM-300M', 'family': 'omni_llm', 'backend': 'omni_external', 'max_train_seconds': 30, 'max_eval_chunk_seconds': 30, 'sample_rate': 16000, 'requires_chat_style': False, 'train_supported_in_this_script': False, 'eval_supported_in_this_script': False, 'notes': 'Omnilingual ASR uses fairseq2/omnilingual-asr workflows. This script exports common JSONL; use the official Meta fine-tuning recipe for actual training.'}
2026-06-26 16:35:59,410 | INFO | Reading jsonl dataset: /home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer/datasets/omnilingual_300m_levantine_full_from_whisper_splits/train.jsonl
2026-06-26 16:35:59,459 | I


## Notes

If this notebook reaches the end, the full dataset has been materialized from the Whisper split sources and exported/prepared for the OmniLingual 300M external recipe flow.

This notebook does **not** run the old smoke manifests and does **not** apply the Whisper `>=30s` filtering rule.
